# Annotation: VEP + CADD

Functional annotation of the variants that converge in the burden test and case-control (file written by the convergent variants notebook), for one ancestry and dataset.

## Setup, imports and configuration

In [ ]:
import os
import json
import time
import requests
import pandas as pd

# Paths
DIR_WSPS  = '/home/jupyter/workspace/ws_files'
DIR_NOVA  = f'{DIR_WSPS}/Novalis_v3_R12'
DIR_RESU  = f'{DIR_NOVA}/Results_union'
DIR_ANNOT = f'{DIR_NOVA}/Annotation'
os.makedirs(DIR_ANNOT, exist_ok=True)

# Ancestry / dataset to annotate
ANC, DS = 'EUR', 'WGS'

# Input: convergent variants written by the convergent variants notebook
VARIANTS_FILE = f'{DIR_RESU}/Convergent_variants_{ANC}_{DS}.tsv'

# Outputs
VEP_RAW_JSON = f'{DIR_ANNOT}/vep_raw_{ANC}_{DS}.json'
ANNOT_FILE   = f'{DIR_ANNOT}/VEP_CADD_annotation_{ANC}_{DS}.tsv'

# VEP REST API
VEP_REST_URL = 'https://rest.ensembl.org/vep/human/region'
HEADERS      = {'Content-Type': 'application/json', 'Accept': 'application/json'}
BATCH_SIZE   = 200   # the API accepts up to 300 variants per request

# CADD API (v1.7, GRCh38)
CADD_BASE = 'https://cadd.gs.washington.edu/api/v1.0/GRCh38-v1.7'

print(f'Annotating {ANC} {DS}')
print(f'  Variants:           {VARIANTS_FILE}')
print(f'  Annotation output:  {ANNOT_FILE}')

## Load convergent variants

In [ ]:
variants = pd.read_csv(VARIANTS_FILE, sep='\t', low_memory=False)
variants['CHROM'] = variants['CHROM'].astype(str)

print(f'Variants loaded: {len(variants)} in {variants["HAR"].nunique()} regions')
variants[['HAR', 'CHROM', 'POS', 'REF', 'ALT', 'OR', 'P', 'FDR_BH', 'sig_tier']]

## Prepare VEP input

In [ ]:
def build_vep_input(df, chrom_col='CHROM', pos_col='POS', ref_col='REF', alt_col='ALT'):
    """
    Convert a variants DataFrame to the VEP REST API input format.
    Returns a list of strings: 'CHR POS POS REF/ALT 1'
    VEP expects 1-based coordinates, strand=1 for forward.
    """
    records = []
    for _, row in df.iterrows():
        chrom = str(row[chrom_col]).replace('chr','')
        pos   = int(row[pos_col])
        ref   = str(row[ref_col])
        alt   = str(row[alt_col])

        # For deletions: start = pos+1, end = pos+len(ref)-1.
        # VEP expects the deleted bases on the REF side of the allele string
        # (deleted/-); the reversed form (-/alt) makes VEP return no result
        # for deletions (e.g. GGA>G needs to be sent as 'GA/-', not '-/G').
        if len(ref) > len(alt):   # deletion
            vep_start  = pos + 1
            vep_end    = pos + len(ref) - 1
            deleted    = ref[len(alt):]   # deleted bases (anchor stripped)
            allele_str = f'{deleted}/-'
        elif len(ref) < len(alt): # insertion
            vep_start = pos
            vep_end   = pos + 1
            allele_str = f'{ref}/{alt}'
        else:                      # SNV
            vep_start = pos
            vep_end   = pos
            allele_str = f'{ref}/{alt}'

        records.append(f'{chrom} {vep_start} {vep_end} {allele_str} 1')
    return records


vep_input = build_vep_input(variants)
print(f'Variants for VEP: {len(vep_input)}')
print(*vep_input[:5], sep='\n')

## VEP annotation via REST API

In [ ]:
def run_vep_rest(variant_list, batch_size=BATCH_SIZE, sleep_between=1.0):
    """
    Call the VEP REST API in batches.
    Returns a list of JSON results (one per variant).
    """
    all_results = []
    batches = [variant_list[i:i+batch_size] for i in range(0, len(variant_list), batch_size)]

    for b_idx, batch in enumerate(batches):
        print(f'  Batch {b_idx+1}/{len(batches)} ({len(batch)} variants)...', end=' ')
        payload = json.dumps({'variants': batch})

        for attempt in range(3):  # retries
            resp = requests.post(VEP_REST_URL, headers=HEADERS, data=payload, timeout=120)
            if resp.status_code == 200:
                data = resp.json()
                all_results.extend(data)
                print(f'OK ({len(data)} results)')
                break
            elif resp.status_code == 429:  # rate limit
                wait = int(resp.headers.get('Retry-After', 5))
                print(f'Rate limit, waiting {wait}s...')
                time.sleep(wait)
            else:
                print(f'ERROR {resp.status_code}: {resp.text[:200]}')
                time.sleep(5)

        if b_idx < len(batches) - 1:
            time.sleep(sleep_between)

    return all_results


def parse_vep_result(result):
    """
    Extract the most relevant fields from a VEP JSON result.
    For non-coding variants, regulatory features and TF motifs are what matters.
    """
    # Variant key built from VEP's 'input' field (same format as add_vep_key)
    _inp   = result.get('input', '')
    _parts = _inp.split()
    var_id = f"{_parts[0]}_{_parts[1]}_{_parts[3]}" if len(_parts) >= 4 else result.get('id', _inp)

    # Overall most severe consequence
    most_severe = result.get('most_severe_consequence', '.')

    # Known variants (rsID, ClinVar)
    colocated = result.get('colocated_variants', [])
    rsid      = next((c['id'] for c in colocated if c.get('id','').startswith('rs')), '.')
    clin_sig  = ';'.join(set(
        cs for c in colocated for cs in c.get('clin_sig', [])
    )) or '.'
    gnomad_af = next(
        (c.get('gnomade_af', c.get('gnomad_af', None))
         for c in colocated if 'gnomade_af' in c or 'gnomad_af' in c),
        None
    )

    # Transcript consequences (nearby genes)
    tc         = result.get('transcript_consequences', [])
    genes      = ';'.join(sorted(set(t.get('gene_symbol','') for t in tc if t.get('gene_symbol')))) or '.'
    tx_conseq  = ';'.join(sorted(set(
        c for t in tc for c in t.get('consequence_terms',[])
    ))) or '.'

    # Regulatory feature consequences (the most important part for HARs)
    reg   = result.get('regulatory_feature_consequences', [])
    reg_biotypes  = ';'.join(sorted(set(r.get('biotype','') for r in reg if r.get('biotype')))) or '.'
    reg_conseqs   = ';'.join(sorted(set(
        c for r in reg for c in r.get('consequence_terms',[])
    ))) or '.'
    reg_feat_ids  = ';'.join(sorted(set(r.get('regulatory_feature_id','') for r in reg if r.get('regulatory_feature_id')))) or '.'
    reg_impact    = ';'.join(sorted(set(r.get('impact','') for r in reg if r.get('impact')))) or '.'

    # TF motif consequences (change in binding affinity)
    motifs        = result.get('motif_feature_consequences', [])
    tf_names      = ';'.join(sorted(set(m.get('motif_name','') for m in motifs if m.get('motif_name')))) or '.'
    tf_score_chg  = ';'.join(
        f"{m.get('motif_name','')}:{m.get('motif_score_change',0):+.3f}"
        for m in motifs if m.get('motif_score_change') is not None
    ) or '.'
    tf_high_inf   = ';'.join(
        m.get('motif_name','') for m in motifs
        if m.get('high_inf_pos') == 'Y'
    ) or '.'  # TFs where it falls on a high-information position (most relevant)

    return {
        'vep_input':       var_id,
        'most_severe':     most_severe,
        'rsID':            rsid,
        'clin_sig':        clin_sig,
        'gnomAD_AF':       gnomad_af,
        'nearest_genes':   genes,
        'transcript_conseq': tx_conseq,
        'reg_biotypes':    reg_biotypes,
        'reg_conseqs':     reg_conseqs,
        'reg_feat_ids':    reg_feat_ids,
        'reg_impact':      reg_impact,
        'TF_names':        tf_names,
        'TF_score_changes': tf_score_chg,
        'TF_high_inf_pos': tf_high_inf,
        '_raw':            result,   # full result, for inspection
    }

### Run VEP

In [ ]:
vep_raw = run_vep_rest(vep_input)
vep_df = pd.DataFrame([parse_vep_result(r) for r in vep_raw])

# Keep the raw JSON in case a field is needed later
with open(VEP_RAW_JSON, 'w') as fh:
    json.dump(vep_raw, fh, indent=2)
print(f'Raw JSON saved: {VEP_RAW_JSON}')
print(f'Parsed results: {len(vep_df)} variants')
vep_df.drop(columns='_raw').head()

## Merge variants with the VEP annotation

In [ ]:
def add_vep_key(df, chrom_col='CHROM', pos_col='POS', ref_col='REF', alt_col='ALT'):
    """Generate the same key that build_vep_input generates, for the merge."""
    keys = []
    for _, row in df.iterrows():
        chrom = str(row[chrom_col]).replace('chr','')
        pos   = int(row[pos_col])
        ref   = str(row[ref_col])
        alt   = str(row[alt_col])

        if len(ref) > len(alt):
            vep_start = pos + 1
            vep_end   = pos + len(ref) - 1
            deleted    = ref[len(alt):]   # same fix as build_vep_input
            allele_str = f'{deleted}/-'
        elif len(ref) < len(alt):
            vep_start = pos
            vep_end   = pos + 1
            allele_str = f'{ref}/{alt}'
        else:
            vep_start = pos
            vep_end   = pos
            allele_str = f'{ref}/{alt}'

        keys.append(f'{chrom}_{vep_start}_{allele_str}')
    return keys


variants['vep_key'] = add_vep_key(variants, chrom_col='CHROM')
vep_df['vep_key']   = vep_df['vep_input'].str.strip()

merged = variants.merge(vep_df.drop(columns=['_raw'], errors='ignore'),
                        on='vep_key', how='left')
print(f'Variants with a VEP result: {merged["vep_input"].notna().sum()} / {len(merged)}')

## Final table

Keeps the case-control, burden and VEP columns.

In [ ]:
cols_final = [
    'HAR', 'CHROM', 'POS', 'REF', 'ALT',
    'sig_tier', 'OR', 'L95', 'U95', 'P', 'FDR_BH', 'A1_FREQ',
    'burden_kernel', 'burden_class', 'burden_p', 'burden_fdr',
    'rsID', 'gnomAD_AF', 'clin_sig',
    'nearest_genes', 'most_severe',
    'reg_biotypes', 'reg_conseqs', 'reg_impact',
    'TF_names', 'TF_score_changes', 'TF_high_inf_pos',
    'reg_feat_ids',
]
final_table = merged[[c for c in cols_final if c in merged.columns]].copy()
print(f'Final table: {len(final_table)} variants')
final_table

## CADD annotation (deleteriousness prediction)

In [ ]:
def get_cadd_scores(df, chrom_col='CHROM', pos_col='POS',
                    ref_col='REF', alt_col='ALT'):
    """
    Query the CADD API v1.7 (GRCh38) one variant at a time.
    Only returns scores for precomputed SNVs; indels may come back as None.
    """
    scores = {}
    for _, row in df.iterrows():
        chrom = str(row[chrom_col])
        pos   = int(row[pos_col])
        ref   = str(row[ref_col])
        alt   = str(row[alt_col])
        url   = f'{CADD_BASE}/{chrom}:{pos}_{ref}_{alt}'
        try:
            resp = requests.get(url, timeout=20)
            if resp.ok:
                data = resp.json()
                if isinstance(data, list) and data:
                    phred = data[0].get('PHRED', None)
                    scores[(chrom, pos, ref, alt)] = phred
                    print(f'  chr{chrom}:{pos} {ref}>{alt} -> CADD_phred = {phred}')
                else:
                    print(f'  chr{chrom}:{pos} {ref}>{alt} -> not precomputed')
            else:
                print(f'  chr{chrom}:{pos} HTTP {resp.status_code}')
        except Exception as e:
            print(f'  Error: {e}')
        time.sleep(0.5)   # respect the rate limit
    return scores


cadd_scores = get_cadd_scores(final_table)

final_table['CADD_phred'] = final_table.apply(
    lambda r: cadd_scores.get(
        (str(r['CHROM']), int(r['POS']), str(r['REF']), str(r['ALT'])), None
    ), axis=1
)
final_table[['HAR', 'POS', 'REF', 'ALT', 'OR', 'CADD_phred']]

## Save results

In [ ]:
final_table.to_csv(ANNOT_FILE, sep='\t', index=False)
print(f'Annotation table saved: {ANNOT_FILE}')